# **Машинное обучение 1, ПМИ ФКН ВШЭ. Бонусное домашнее задание**
##  **Основные методы оценки важности признаков для ML моделей**

В этом домашнем задании вы будете работать с данными о Всемирном отчёте о счастье (World Happiness Report). В качестве теории используйте следующие семинарские материалы: 
- [Основные методы оценки важности признаков для ML моделей](https://github.com/esokolov/ml-course-hse/blob/master/2025-fall/seminars/sem11-xai.ipynb);
- [Интерпретация по определению](https://github.com/esokolov/ml-course-hse/blob/master/2025-fall/seminars/sem11-xai.pdf).

**Общая информация**

**Дата выдачи:** 04.05.2026

**Дедлайн:** 18.05.2026 23:59MSK

**Оценивание и штрафы**

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимальная оценка за работу - 5 баллов.

Как всегда, сдача после жёсткого дедлайна невозможна. При выставлении неполного балла за задание в связи с наличием ошибок на усмотрение проверяющего предусмотрена возможность исправить работу на указанных в ответном письме условиях.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).


**NB** В этом задании много работ, требующих вывода и размышлений. Куцые выводы, как и неэффективная реализация кода, могут негативно отразиться на оценке.



**Задания сдаются через систему anytask.** Посылка должна содержать:

Ноутбук homework-practice-xai-Username.ipynb
Username - ваша фамилия и имя на латинице именно в таком порядке.


## **Подготовительная часть**

In [ ]:
!pip install shap -qU

In [ ]:
!pip install -qU pyALE

In [ ]:
!pip install -qU lime fat-forensics[all]

### **Подготовка данных**

Загрузите датасет по ссылке.

In [ ]:
# import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

# import pandas as pd
import polars as pl
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import partial_dependence
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler


In [ ]:
layout_dict = {
    "margin": {"l": 20, "r": 20, "t": 40, "b": 20},
    "width": 600,
    "height": 400,
    "paper_bgcolor": "LightSteelBlue",
    "title_font_size": 14,
    "xaxis_title_font_size": 12,
    "yaxis_title_font_size": 12,
}

In [ ]:
!export KAGGLE_API_TOKEN=$(cat /data/config/.kaggle/token) && kaggle datasets download --help

In [ ]:
!mkdir -p /data/ml-course-hse/ml1-2026-spring/homework-practice-bonus-XAI && export KAGGLE_API_TOKEN=$(cat /data/config/.kaggle/token) && kaggle datasets download ajaypalsinghlo/world-happiness-report-2021 -p /data/ml-course-hse/ml1-2026-spring/homework-practice-bonus-XAI --unzip

In [ ]:
!wget  -O '/data/ml-course-hse/ml1-2026-spring/homework-practice-bonus-XAI/world-happiness-report-2021-prep.csv' -q 'https://www.dropbox.com/scl/fi/vn5d4awcg7n307bnewr2x/world-happiness-report-2021-prep-1.csv?rlkey=yk8sn3fblpvtu8iyc02vpryhp&st=cvysdgto&dl=0'

In [ ]:
data_to_work = pl.read_csv(
    "/data/ml-course-hse/ml1-2026-spring/homework-practice-bonus-XAI/world-happiness-report-2021-prep.csv"
)

data_to_work.head(5)

### Примечание про столбец Dystopia + residual.


В World Happiness Report Dystopia - это не реальная страна, а техническая точка отсчёта: гипотетическая страна с минимальными значениями по ключевым факторам (GDP, social support, life expectancy, freedom, generosity, corruption). Её вводят как бенчмарк, чтобы вклады факторов в разложении были неотрицательными.

Столбец `Dystopia + residual` - это  разложения: базовый уровень (оценка жизни в Dystopia) плюс остаточная часть, которую шесть факторов не объясняют для конкретной страны (ошибка/невязка модели). Это не социальный индекс, а слагаемое, которое вместе с вкладами 6 факторов даёт итоговый score.

Иначе говоря, в таблицах WHR:
Score ~ (вклады 6 факторов) + (Dystopia + residual).

### Поехали дальше

In [ ]:
X = data_to_work.drop("Ladder score")
y = data_to_work["Ladder score"]

Для обучения будем использовать два способа масштабирования - `MinMaxScaler` и `StandardScaler`.

In [ ]:
feature_names = list(X.columns)

# Выберем признак для анализа
feature_idx = 1
feature = X.columns[feature_idx]

print(f"FEATURE TO ANALYZE: {feature}")

In [ ]:
# Разделение на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# 1
sc_minmax = MinMaxScaler()
X_train_sc_minmax = sc_minmax.fit_transform(X_train)
X_test_sc_minmax = sc_minmax.transform(X_test)

# 2
sc_standard = StandardScaler()
X_train_sc_standard = sc_standard.fit_transform(X_train)
X_test_sc_standard = sc_standard.transform(X_test)

### **Задание 1. (0.5 балла)**
- Обучите 6 моделей:
  -  линейную регрессию (`LinearRegression`) на двух вариантах данных;
  - Lasso регрессию (`Lasso`) на двух вариантах данных;
  - градиентный бустинг (`GradientBoostingRegressor`) на двух вариантах данных;
- Выведите MSE и RMSE моделей.
- Зафиксируйте выводы.

> Цель задания - увидеть, какие модели чувствительны к масштабу признаков (Lasso), а какие почти инвариантны (ls/деревья), и почему это важно для интерпретации.


**NB:** Для бустингов ограничьте глубину до 5.

In [ ]:
res = pl.DataFrame(
    schema={
        "model": pl.String,
        "mse_train": pl.Float64,
        "rmse_train": pl.Float64,
        "mse_test": pl.Float64,
        "rmse_test": pl.Float64,
    }
)

In [ ]:
lr_mm = LinearRegression()
lr_mm.fit(X_train_sc_minmax, y_train)
mse_train = mean_squared_error(y_train, lr_mm.predict(X_train_sc_minmax))
mse_test = mean_squared_error(y_test, lr_mm.predict(X_test_sc_minmax))

res.extend(
    pl.from_dicts(
        [
            {
                "model": "lr_mm",
                "mse_train": mse_train,
                "rmse_train": mse_train**0.5,
                "mse_test": mse_test,
                "rmse_test": mse_test**0.5,
            }
        ]
    )
)

lr_st = LinearRegression()
lr_st.fit(X_train_sc_standard, y_train)
mse_train = mean_squared_error(y_train, lr_st.predict(X_train_sc_standard))
mse_test = mean_squared_error(y_test, lr_st.predict(X_test_sc_standard))

res.extend(
    pl.from_dicts(
        [
            {
                "model": "lr_st",
                "mse_train": mse_train,
                "rmse_train": mse_train**0.5,
                "mse_test": mse_test,
                "rmse_test": mse_test**0.5,
            }
        ]
    )
)

In [ ]:
ls_mm = Lasso(alpha=1e-4)
ls_mm.fit(X_train_sc_minmax, y_train)
mse_train = mean_squared_error(y_train, ls_mm.predict(X_train_sc_minmax))
mse_test = mean_squared_error(y_test, ls_mm.predict(X_test_sc_minmax))

res.extend(
    pl.from_dicts(
        [
            {
                "model": "ls_mm",
                "mse_train": mse_train,
                "rmse_train": mse_train**0.5,
                "mse_test": mse_test,
                "rmse_test": mse_test**0.5,
            }
        ]
    )
)

ls_st = Lasso(alpha=1e-4)
ls_st.fit(X_train_sc_standard, y_train)
mse_train = mean_squared_error(y_train, ls_st.predict(X_train_sc_standard))
mse_test = mean_squared_error(y_test, ls_st.predict(X_test_sc_standard))

res.extend(
    pl.from_dicts(
        [
            {
                "model": "ls_st",
                "mse_train": mse_train,
                "rmse_train": mse_train**0.5,
                "mse_test": mse_test,
                "rmse_test": mse_test**0.5,
            }
        ]
    )
)

In [ ]:
gb_mm = GradientBoostingRegressor(
    loss="squared_error",
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    learning_rate=1e-1,
)
gb_mm.fit(X_train_sc_minmax, y_train)
mse_train = mean_squared_error(y_train, gb_mm.predict(X_train_sc_minmax))
mse_test = mean_squared_error(y_test, gb_mm.predict(X_test_sc_minmax))

res.extend(
    pl.from_dicts(
        [
            {
                "model": "gb_mm",
                "mse_train": mse_train,
                "rmse_train": mse_train**0.5,
                "mse_test": mse_test,
                "rmse_test": mse_test**0.5,
            }
        ]
    )
)

gb_st = GradientBoostingRegressor(
    loss="squared_error",
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=10,
    learning_rate=1e-1,
)
gb_st.fit(X_train_sc_standard, y_train)
mse_train = mean_squared_error(y_train, gb_st.predict(X_train_sc_standard))
mse_test = mean_squared_error(y_test, gb_st.predict(X_test_sc_standard))

res.extend(
    pl.from_dicts(
        [
            {
                "model": "gb_st",
                "mse_train": mse_train,
                "rmse_train": mse_train**0.5,
                "mse_test": mse_test,
                "rmse_test": mse_test**0.5,
            }
        ]
    )
)

## **1. Интерпретация по определению**

Воспользуемся встроенными важностями в моделях. Для бустинга — усредненная важность по деревьям, для регрессий — коэффициенты.

### **Задание 2. (0.25 балла)**
- Постройте график (любой, на ваш выбор), позволяющий визуально оценить и сравнить коэффициенты по парам моделей:
  - Модель и `StandardScaler` vs Модель и `MinMaxScaler`
- Зафиксируйте выводы по каждой из пар графиков.

In [ ]:
features = sc_minmax.get_feature_names_out()

fig = px.bar(
    pl.concat(
        [
            pl.DataFrame(
                {
                    "feature": features,
                    "coefficient": lr_mm.coef_,
                }
            ).with_columns(model=pl.lit("lr_mm")),
            pl.DataFrame(
                {
                    "feature": features,
                    "coefficient": lr_st.coef_,
                }
            ).with_columns(model=pl.lit("lr_st")),
        ]
    ),
    x="feature",
    y="coefficient",
    color="model",
    barmode="group",
)


fig.update_layout(**layout_dict)

**Ваш вывод здесь:**

Standard scaler shrink features less aggressively, so coefficients are smaller

In [ ]:
# Ваш код здесь — Lasso + два варианта

features = sc_minmax.get_feature_names_out()

fig = px.bar(
    pl.concat(
        [
            pl.DataFrame(
                {
                    "feature": features,
                    "coefficient": ls_mm.coef_,
                }
            ).with_columns(model=pl.lit("lasso_mm")),
            pl.DataFrame(
                {
                    "feature": features,
                    "coefficient": ls_st.coef_,
                }
            ).with_columns(model=pl.lit("lasso_st")),
        ]
    ),
    x="feature",
    y="coefficient",
    color="model",
    barmode="group",
)


fig.update_layout(**layout_dict)

**Ваш вывод здесь:**

With small regularization parameter, pretty similar to Linear Regression results

In [ ]:
# Ваш код здесь — Lasso + два варианта

features = sc_minmax.get_feature_names_out()

fig = px.bar(
    pl.concat(
        [
            pl.DataFrame(
                {
                    "feature": features,
                    "importance": gb_mm.feature_importances_,
                }
            ).with_columns(model=pl.lit("gb_mm")),
            pl.DataFrame(
                {
                    "feature": features,
                    "importance": gb_st.feature_importances_,
                }
            ).with_columns(model=pl.lit("gb_st")),
        ]
    ),
    x="feature",
    y="importance",
    color="model",
    barmode="group",
)


fig.update_layout(**layout_dict)

**Ваш вывод здесь:**


Identical for both scalers as is expected, broadly the conclusions are similar to LR results

### **Задание 3. (0.25 балла)**
Вернитесь к графику линейной регрессии. Очевидно, что коэффициенты одной модели связаны с коэффициентами другой. Как перейти от одних к другим? Реализуйте это в коде, как для свободного члена, так и для самих коэффициентов.

> Идея: коэффициенты линейной модели можно сравнивать как важности только при фиксированном масштабе. Вывод формулы перехода - способ строго показать, что коэффициенты меняются предсказуемо и не являются абсолютной важностью.

**Ваши теоретические выкладки здесь:**

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

coef_std = lr.coef_ * sc_standard.scale_
intercept_std = lr.intercept_ + np.dot(lr.coef_, sc_standard.mean_)

assert round(intercept_std, 4) == round(lr_st.intercept_, 4)
assert all(np.round(coef_std, 4) == np.round(lr_st.coef_, 4))

## **1. Plot-based методы**

Рассмотрев тонкости работы с масштабированием, поработаем с plot-based методами. Оставьте теперь для анализа линейную регрессию и бустинг, обученные на MinMaxScaler.

In [ ]:
lr_minmax = lr_mm
gb_minmax = gb_mm

### **1. 1. ICE**

**Построение:**

1. Зафиксируем множество $X_{test}$ и некоторый признак $j$. Пусть  $j$ имеет $m$ уникальных значений $[j_1, j_2, ..., j_m]$.

2. Исходный датасет  $X_{test}$ дублируем  $m$ раз: $$X'_1, X'_2, ..., X'_m,$$ так, что для датасета $X'_i$ значение признака $j$ есть $j_i$;

3. На каждом $X'_i$ рассчитываем прогноз модели $f(X'_i)$&

4. На графике строим линии для каждого объекта, показывающие как прогноз (ось $y$) меняется при изменении признака (ось $x$);

### **Задание 4. (0.5 балла)**

Мы знаем, что признаки модели масштабированы определенным образом. Минимум каждого признака равен 0, максимум 1. Что будет с моделями, если признаки выйдут из диапазона?
- Работая с тем же признаком, изучите его **исходные** минимум и максимум. Сделайте искусственный диапазон из 20 точек, равный $[max, 3*max]$.
- На полученном диапазоне постройте ICE для обеих моделей. ICE стройте собственноручно, без использования sklearn.
- Зафиксируйте выводы


In [ ]:
feature_idx = 1
feature = feature_names[feature_idx]

X_ice = pl.DataFrame(X_train, schema=feature_names)
rng = list(X_ice[feature].sort())
rng.extend([rng[-1] * i / 10 for i in range(11, 31)])

X_ice = pl.concat([X_ice.with_columns(pl.lit(v).alias(feature)) for v in rng])

X_ice_mm = sc_minmax.transform(X_ice)
X_ice = pl.concat(
    [X_ice, pl.DataFrame(X_ice_mm, schema=[f + "_mm" for f in feature_names])],
    how="horizontal",
)
X_ice = X_ice.with_columns(
    lr_pred=lr_minmax.predict(X_ice_mm), gb_pred=gb_minmax.predict(X_ice_mm)
)

X_ice = X_ice.with_row_index(name="ix").with_columns(pl.col("ix").mod(len(X_train)))


In [ ]:
fig = px.line(
    X_ice,
    x=feature,
    y="lr_pred",
    line_group="ix",
)

fig.update_layout(**layout_dict)
fig.update_traces(opacity=0.4)

In [ ]:
fig = px.line(
    X_ice,
    x=feature,
    y="gb_pred",
    line_group="ix",
)

fig.update_layout(**layout_dict)
fig.update_traces(opacity=0.4)

**Ваш вывод здесь:**

Gradient Boosting is flat outside of the range, while LR is identically linear with the same slope

### **1. 2. PDP**

Теперь посмотрим на усредненнуе влияние признака.
### **Задание 5. (0.25 балла)**

- Усредните ICE по обеим моделям, используя PDP из sklearn.
- Сделайте выводы.


In [ ]:
results_avg = partial_dependence(
    gb_minmax,
    X_train_sc_minmax,
    [feature_idx],
    grid_resolution=20,
    kind="average",
    method="brute",
)

In [ ]:
results = partial_dependence(
    gb_minmax,
    X_train_sc_minmax,
    [feature_idx],
    grid_resolution=20,
    kind="individual",
)

In [ ]:
fig = px.line(
    pl.concat(
        [
            pl.DataFrame({"feature": results.grid_values[0], "pred": y}).with_columns(
                ix=pl.lit(i), kind=pl.lit("individual")
            )
            for i, y in enumerate(results.individual[0], 1)
        ]
        + [
            pl.DataFrame(
                {"feature": results_avg.grid_values[0], "pred": results_avg.average[0]}
            ).with_columns(ix=pl.lit(0), kind=pl.lit("average"))
        ]
    ),
    x="feature",
    y="pred",
    line_group="ix",
    color="kind",
)

fig.update_layout(**layout_dict)
fig.update_traces(opacity=0.4)

**Ваш вывод здесь:**

### **1. 3. Accumulated Local Effects (ALE)**

Заключительный из основных графических методов — ALE. Ради практики построим и его.

### **Задание 6. (0.5 балла)**

- Постройте ALE по обеим моделям, используя pyALE. (0.15)
- Подберите размер сетки так, чтобы получить доверительные интервалы. (0.05)
- Проанализируйте полученный график. Каковы получились ДИ? Почему они различны для моделей? (0.3)

> Важно: сетку значений строим в исходных единицах признака, но перед подачей в модель применяем тот же scaler, на котором обучение.

In [ ]:
from PyALE import ale
from sklearn.pipeline import Pipeline

In [ ]:
pipeline = Pipeline(steps=[("scaler", sc_minmax), ("regression", gb_minmax)])

In [ ]:
feature_idx = 2
ale_eff = ale(
    X_train.to_pandas(),
    model=pipeline,
    feature=[feature_names[feature_idx]],
    grid_size=20,
)

**Ваши выводы здесь:**

## **2. Permutation importances**

Практикуем ещё один метод.
### **Задание 7. (0.25 балла)**

- Постройте Permutation importances по обеим моделям, используя sklearn
- Поэкспериментируйте с числом перестановок. (0.05)
- Проанализируйте полученные коэффициенты. Как они меняются от количества перестановок? Как меняются std коэффициентов? Зафиксируйте выводы. (0.20)

In [ ]:
from sklearn.inspection import permutation_importance

# Ваш код здесь

In [ ]:
perm_results = permutation_importance(
    lr_minmax,
    X_test_sc_minmax,
    y_test,
    scoring="neg_mean_squared_error",
    n_repeats=400,
    random_state=17,
)

In [ ]:
fig = px.box(
    pl.concat(
        pl.DataFrame({"importance": imp}).with_columns(feature=pl.lit(f))
        for f, imp in zip(feature_names, perm_results.importances)
    ),
    x="feature",
    y="importance",
)

fig.update_layout(**layout_dict)
fig.update_layout({"height": 600, "width": 800})

**Ваши выводы здесь:**

The standard deviation and the mean of feature importance don't depend much on the number of permuatations as long as it's sufficiently high

### **Задание 8. (0.5 балла)**

Идея перестановочной важности представляет собой частный случай важности при помощи внесения **возмущений** в признак. Возмущения могут быть разных видов:

- внесение случайного шума;
- зануление признака (удобно для изображений);
- сдвиг признака к его базовому значению и оценка траектории изменения прогнозов или качества модели;

В этом задании попробуем построить оценку на основе сдвига к базовому значению траектории. Примем за базовое значение медиану признака и будем сдвигать исходный признак к медианному с некоторым коэффициентом $\beta$:

$$x_j^{(\beta)}=(1-\beta)x_j+\beta b_j$$

где $b_j$ – медиана признака. Важность будем оценивать как изменение MSE по тест-данным.

1. Реализуйте это возмущение. Как меняются важности при разных $\beta$?
2. Постройте bar-график и сравните ранжирование с **permutation importance**. При сравнении, анализируйте только числовые признаки.


Для регрессии эффект очевиден, поэтому эту окклюзию оценивайте только для бустинга.
**Подсказка:**

1. Выберите базовые значения $x_j$ как **медианы по train** (базовые значения обозначим за $b_j$);
2. На тренировочном наборе данных для каждого признака $j$ замените столбец $x_j$ на $x^{(\beta)}_j=(1-\beta)x_j+\beta b_j$ при разных $\beta$, равных $[0.2, 0.5, 1]$.
3. Посчитайте $\Delta L_j = \text{MSE}(y,\hat y^{(\beta)}_j)-\text{MSE}(y,\hat y)$. Чем больше $\Delta L_j$, тем важнее признак.



In [ ]:
def occlusion_importance(beta):
    mse = mean_squared_error(y_test, gb_minmax.predict(X_test_sc_minmax))
    res = []
    for i in range(len(feature_names)):
        X_temp = X_test_sc_minmax.copy()
        X_temp[:, i] = (
            X_temp[:, i] * (1.0 - beta) + np.median(X_train_sc_minmax[i]) * beta
        )
        mse_beta = mean_squared_error(y_test, gb_minmax.predict(X_temp))
        res.append(mse_beta - mse)

    return pl.DataFrame({"feature": feature_names, "importance": res}).with_columns(
        beta=pl.lit(beta)
    )

In [ ]:
fig = px.bar(
    pl.concat(occlusion_importance(beta) for beta in [0.2, 0.5, 1.0]).with_columns(
        pl.col("beta").cast(pl.String)
    ),
    x="feature",
    y="importance",
    color="beta",
    barmode="group",
)

fig.update_layout(**layout_dict)

**Ваши выводы здесь:**

When beta is small, the difference is not very pronounced. Otherwise, conclusions are similar to permutation importance, but seem less granular (e.g. no distribution across disturbances)

## **3. SHAP**

Теперь проведем локальный анализ, используя SHAP и LIME. Но для shap также построим глобальные графики.

### **Задание 9. (0.5 баллa)**
- Постройте два глобальных графика, используя SHAP. Как один из них обязательно используйте `force`, как другой — выберите любой из документации. (0.25 балл)
- Проанализируйте важность признаков в обоих алгоритмах.(0.25 балл)

**NB:** Для линейного алгоритма используйте masker=`shap.maskers.Independent(data=X_train_sc)`. Он имитирует отсутствие признака

In [ ]:
import shap
# Ваш код здесь

In [ ]:
# shap.initjs()
explainer_gb = shap.TreeExplainer(gb_minmax, feature_names=feature_names)
shap_values_gb = explainer_gb(X_train_sc_minmax)
shap.plots.force(shap_values_gb)

In [ ]:
feature_names

In [ ]:
shap.plots.force(shap_values_gb[7, :])

In [ ]:
# shap.initjs()
explainer_lr = shap.LinearExplainer(
    lr_minmax,
    feature_names=feature_names,
    masker=shap.maskers.Independent(
        data=X_train_sc_minmax, max_samples=len(X_train_sc_minmax)
    ),
)
shap_values_lr = explainer_lr(X_train_sc_minmax)
shap.plots.force(shap_values_lr)

 **Ваш вывод здесь:**

Kinda difficult to assess relative feature importance from the global chart... But can look at mean absolute Shapley values

In [ ]:
shap.plots.bar(shap_values_gb)

In [ ]:
shap.plots.bar(shap_values_lr)

## **Локальное объяснение.**

### **Задание 10. (0.25 балла)**
- Постройте локальный график с SHAP для объекта с индексом 7 на обеих моделях и сделайте выводы.

In [ ]:
y_pred = gb_minmax.predict(X_train_sc_minmax)
assert round(
    shap_values_gb[7, :].values.sum() + shap_values_gb[7, :].base_values, 4
) == round(y_pred[7], 4)

In [ ]:
shap.plots.force(shap_values_gb[7, :])

In [ ]:
shap.plots.force(shap_values_lr[7, :])

**Ваш вывод здесь:**

### **Теория: SHAP + categorical**


SHAP разлагает предсказание модели $f(x)$ на базовый уровень $\phi_0=\mathbb E[f(X)]$ и сумму вкладов признаков:

$$
f(x)=\phi_0+\sum_{j=1}^p \phi_j(x).
$$

Значение $\phi_j(x)$ — это «маржинальный вклад» признака $j$ вблизи точки $x$, усреднённый по всем коалициям признаков. Сумма SHAP по признакам всегда совпадает с предсказанием относительно базовой линии — отсюда удобство интерпретации и сравнимость вкладов.

Однако, вопреки этому удобству, SHAP имеет ограничения при работе с категориальными данными в разных кодировках.


1. OneHotEncoding. В этом случае каждому столбцу присвается свое значение Шепли. Общий вклад здесь — всегда **сумма** dummy-столбцов.
2. В CatBoost используется особый способ категориального разбиения, который (при использовании) фактически создаёт новые признаки для разбиения, отсутствующие в исходном наборе входных признаков. Эти признаки позволяют разбивать целые группы категорий тем или иным образом. Поддержка таких типов деревьев пока не реализована в SHAP, но можно использовать в CatBoost — встроенные [ShapValues](https://catboost.ai/docs/en/concepts/shap-values).
3. При других типах кодировок важно проверять отсутствие утечки.



## **4. LIME**

### **Задание 11. (0.25 балла)**
- Постройте локальный график с LIME для объекта с индексом 7 на обеих моделях и сделайте выводы. Сравните графики с SHAP. Ядро используйте как в семинаре — `kernel_width=np.sqrt(len(feature_names)) * 0.75`

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# Ваш код здесь

In [ ]:
# Ваш код здесь

### **Задание 12. (1 балл)**

Теперь сделаем LIME собственноручно. Ваша реализация должна:

- Принимать на вход точку `x0`
- Вокруг этой точки данных сэмплировать окрестность заданного размера `n`
- На данных из этой окрестности получать прогнозы объясняемой модели (учитывать веса объектов, используя ядро)
- Обучать LIME модель в виде стандартной линейной регрессии
- Возвращать коэффициенты локальной модели


Для своей реализации оцените устойчивость. Как влияет на коэффициенты количество сгенерированных точек? А выбор ядра?

**В реализации вам может помочь библиотека `fatf`**
**Свой LIME тестируйте только для бустинга.**

In [ ]:
# Ваш код здесь — функция

In [ ]:
# Ваш код здесь применение на бустинге

 **Ваш вывод здесь:**

## **Вы проделали огромную работу! Ура!**